In [ ]:
#!pip install ddgs trafilatura
#!pip install lxml_html_clean



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint

from ddgs import DDGS
import trafilatura

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

client = OpenAI()

#List of tools we will use, e.g. search_web and fetch_url functions
tools = []

### Step 1) Define the Tools

In [ ]:
def search_web(query: str):
    """Search the web using DuckDuckGo browser. Returns 3 results."""
    ddgs = DDGS()
    results = ddgs.text(query, max_results = 3)
    print(f"  \u2705 Got results")
    return json.dumps(results, indent=2)

# #Test code
# query = "new season of Solo Leveling 2 on crunchy roll" 
# search_web(query)


In [ ]:
def fetch_url(url: str):
    """Fetch te content of a URL using trafilatura."""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f". \u2705 Got text: {len(text)} chars")
            return text
    print(f". \u274C Failed to fetch or extract text from {url}")
    return f"Could not extract text from {url}. Try a different source"

# #Test code
# url = "https://en.wikipedia.org/wiki/Naruto_Uzumaki"
# url = "https://www.linkedin.com/jobs/view/4366922049/"
# fetch_url(url)

### Step 2) Describe the functions

In [20]:

search_web_function = {
    "name": "searc_web",
    "description": "Search the web using DuckDuckGo browser. Returns 3 results.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to find relevant websites"
            }
        },
        "required": ["query"]
    }
}

tools.append({"type": "function", "function": search_web_function})

In [21]:
fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch the content of a URL using trafilatura.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The web address to be fetched by the function"
            }
        },
        "required": ["url"]
    }
}

tools.append({"type": "function", "function": fetch_url_function})

### Step 3) Tool Call Handler

In [24]:
#------ Tool Handler -----------
def handle_tool_call(tool_calls):
    tool_results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f"Calling fucnction {function_name}") #For future debuggins

        if function_name == 'search_web':
            result = search_web(args['query'])
            content = f"Search results: {result}"
        elif function_name == 'fetch_url':
            result = fetch_url(args['url'])
            content = f"Fetched URL content: {result}"
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }
        tool_results.append(tool_call_result)
    return tool_results



### Step 4) The System Prompt

This tells the LLM **who it is and how to behave**. The key things:  
* What its job is
* What tools it has
* What process to follow
* Wat output format to produce

In [ ]:
RESEARCH_AGENT_PROMPT = """
You are an expert research assistant agent. Your primary tasks is to understand what the user wants to know, ask follow up questions if you are unclear about the user's desired topic, and most importantly to use the tools at your disposal to help the user learn about their desired topic as thoroughly as possible.

You have the following tools you can use:
- search_web, a function that searches the web given a query
- fetch_url, a function that fetches a given url and return the results

When interacting with the user, you can ask questions to understand WHAT the user wants to know, HOW MUCH detail they want to know about it, and if they have a PREFERENCE for the types of web sources that should be used as sources; the user can potentially fail to provide a response, in which case you can make an educated guess about those things and be transparent about your assumptions by telling the user. 

Once you have an idea of what to look for, do a web search for that information (potentially prioritizing types of sources given the type of information requested). Combine information from several different websites, if multiple websites have relevant info. Condense and highlight what the user wants to know from that overall aggregation. 

Your response to the user should be warm, approachable, transparent about your approach and findings, and thorough. Always give an Executive Summary with the most important details at the very end of the response. 

"""

### Step 5) The Agentic Loop